In [2]:
# ==========================================
# Experiment 7: Self-Training
# Semi-Supervised Learning
# ==========================================

# 1. Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report
)


# ==========================================
# 2. Load Dataset
# ==========================================

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset Shape:", X.shape)
print("Class Names:", data.target_names)


# ==========================================
# 3. Split Dataset
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ==========================================
# 4. Feature Scaling
# ==========================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==========================================
# 5. Select Labeled and Unlabeled Data
# ==========================================

# Select 20% labeled data
# Remaining 80% will be unlabeled

indices = np.arange(len(X_train))

labeled_indices, unlabeled_indices = train_test_split(
    indices,
    test_size=0.8,
    random_state=42,
    stratify=y_train
)

# Create semi-supervised labels
y_semi = np.full(len(y_train), -1)

# Assign actual labels to labeled samples
y_semi[labeled_indices] = y_train[labeled_indices]

print("\nDataset Information")
print("-------------------------")
print("Total Training Samples:", len(X_train))
print("Labeled Samples:", len(labeled_indices))
print("Unlabeled Samples:", len(unlabeled_indices))


# ==========================================
# 6. Create Base Classifier
# ==========================================

base_model = SVC(
    probability=True,
    random_state=42
)


# ==========================================
# 7. Apply Self-Training
# ==========================================

self_training = SelfTrainingClassifier(
    estimator=base_model,
    threshold=0.8,
    max_iter=10,
    verbose=True
)


# Train using labeled and unlabeled data
self_training.fit(X_train, y_semi)


# ==========================================
# 8. Generate Predictions
# ==========================================

y_pred = self_training.predict(X_test)


# ==========================================
# 9. Evaluation
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("\nEvaluation Results")
print("-------------------------")
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=data.target_names
    )
)


# ==========================================
# 10. Pseudo-Label Information
# ==========================================

# Count samples that received pseudo-labels
pseudo_labeled = np.sum(
    self_training.transduction_[unlabeled_indices] != -1
)

print("\nSelf-Training Information")
print("-------------------------")
print("Initial Labeled Samples:", len(labeled_indices))
print("Initial Unlabeled Samples:", len(unlabeled_indices))
print("Pseudo-Labeled Samples:", pseudo_labeled)
print("Final Labeled Samples:",
      np.sum(self_training.transduction_ != -1))
print("Number of Iterations:", self_training.n_iter_)

Dataset Shape: (569, 30)
Class Names: ['malignant' 'benign']

Dataset Information
-------------------------
Total Training Samples: 455
Labeled Samples: 91
Unlabeled Samples: 364
End of iteration 1, added 319 new labels.
End of iteration 2, added 25 new labels.
End of iteration 3, added 2 new labels.
End of iteration 4, added 2 new labels.
End of iteration 5, added 1 new labels.

Evaluation Results
-------------------------
Accuracy: 0.9736842105263158

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.95      0.96        42
      benign       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114


Self-Training Information
-------------------------
Initial Labeled Samples: 91
Initial Unlabeled Samples: 364
Pseudo-Labeled Samples: 349
Final Labeled Samples: 440
Number of Iterations: 6
